# Task 1: Compare native document-processing outputs

Process the same two-page claim statement with three approaches and inspect what each service actually returns.

| Approach | Requested output |
|---|---|
| Configured multimodal GPT model | Structured claim JSON directly from both images |
| Mistral Document AI | OCR Markdown for each image |
| Azure Document Intelligence | Layout Markdown for each image |

There is deliberately no shared normalization model after OCR. The goal is to see the real differences before deciding what downstream processing each approach needs.

## 1. Load configuration

Select the repository's Python environment as the notebook kernel. The `.env` file must have been generated in Challenge 0.

In [ ]:
from __future__ import annotations

import base64
import os
import sys
import time
from pathlib import Path
from typing import Optional

from dotenv import load_dotenv
from IPython.display import Image, JSON, Markdown, display
from pydantic import BaseModel, ConfigDict


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "challenge-0").is_dir() and (candidate / "challenge-1").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the claims-processing-hack repository.")


REPO_ROOT = find_repo_root()
load_dotenv(REPO_ROOT / ".env")

CLAIM_ID = "crash1"
STATEMENTS_DIR = REPO_ROOT / "challenge-0" / "data" / "statements"
FRONT_IMAGE = STATEMENTS_DIR / f"{CLAIM_ID}_front.jpeg"
BACK_IMAGE = STATEMENTS_DIR / f"{CLAIM_ID}_back.jpeg"

required_variables = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_KEY",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
    "MISTRAL_DOCUMENT_AI_ENDPOINT",
    "MISTRAL_DOCUMENT_AI_KEY",
    "MISTRAL_DOCUMENT_AI_DEPLOYMENT_NAME",
    "DOCUMENT_INTELLIGENCE_ENDPOINT",
    "DOCUMENT_INTELLIGENCE_KEY",
]
missing_variables = [name for name in required_variables if not os.getenv(name)]
if missing_variables:
    raise RuntimeError(f"Missing variables in .env: {', '.join(missing_variables)}")

print(f"Repository: {REPO_ROOT}")
print(f"Claim: {CLAIM_ID}")
print(f"Multimodal deployment: {os.environ['AZURE_OPENAI_DEPLOYMENT_NAME']}")

## 2. Inspect the source document

Read both pages yourself before inspecting model output. Pay particular attention to identifiers such as the policy number, VIN, license plate, phone numbers, and police report number.

In [ ]:
display(Markdown("### Front page"))
display(Image(filename=str(FRONT_IMAGE), width=700))
display(Markdown("### Back page"))
display(Image(filename=str(BACK_IMAGE), width=700))

## 3. Multimodal GPT: structured output

The GPT approach performs document understanding and semantic field extraction in one request. We explicitly ask it for a typed claim object.

In [ ]:
from openai import AzureOpenAI


class ClaimStatement(BaseModel):
    model_config = ConfigDict(extra="forbid")

    claimant_id: Optional[str] = None
    policy_holder_name: Optional[str] = None
    policy_holder_address: Optional[str] = None
    policy_holder_phone: Optional[str] = None
    policy_holder_email: Optional[str] = None
    policy_number: Optional[str] = None
    vehicle_year_make_model: Optional[str] = None
    vehicle_color: Optional[str] = None
    vehicle_vin: Optional[str] = None
    vehicle_license_plate: Optional[str] = None
    incident_date: Optional[str] = None
    incident_time: Optional[str] = None
    incident_location: Optional[str] = None
    incident_description: Optional[str] = None
    damage_description: Optional[str] = None
    witness_name: Optional[str] = None
    witness_phone: Optional[str] = None
    police_department: Optional[str] = None
    police_report_number: Optional[str] = None
    officer_name: Optional[str] = None
    repair_shop_name: Optional[str] = None
    repair_shop_address: Optional[str] = None
    claim_request: Optional[str] = None
    signature_name: Optional[str] = None
    signature_date: Optional[str] = None


def image_data_url(path: Path) -> str:
    encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"


openai_client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

started = time.perf_counter()
completion = openai_client.beta.chat.completions.parse(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    messages=[
        {
            "role": "system",
            "content": (
                "Extract the insurance claim statement into the supplied schema. "
                "Use only information explicitly visible in the two images. "
                "Use null for missing, illegible, or uncertain fields."
            ),
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Extract this two-page claim statement."},
                {"type": "image_url", "image_url": {"url": image_data_url(FRONT_IMAGE)}},
                {"type": "image_url", "image_url": {"url": image_data_url(BACK_IMAGE)}},
            ],
        },
    ],
    response_format=ClaimStatement,
    max_tokens=4000,
)
gpt_result = completion.choices[0].message.parsed
if gpt_result is None:
    raise RuntimeError(f"The model did not return a claim: {completion.choices[0].message.refusal}")
gpt_seconds = time.perf_counter() - started

display(JSON(gpt_result.model_dump()))
print(f"Returned {sum(value is not None for value in gpt_result.model_dump().values())} populated fields")
print(f"Completed in {gpt_seconds:.2f} seconds")

## 4. Mistral Document AI: OCR Markdown

Mistral processes each page independently and returns Markdown. No GPT model restructures or corrects this text afterward.

In [ ]:
statements_processing_dir = REPO_ROOT / "challenge-1" / "statements_processing"
if str(statements_processing_dir) not in sys.path:
    sys.path.insert(0, str(statements_processing_dir))

from mistral_doc_intelligence import get_ocr_results

started = time.perf_counter()
mistral_front = get_ocr_results(str(FRONT_IMAGE))
mistral_back = get_ocr_results(str(BACK_IMAGE))
mistral_seconds = time.perf_counter() - started
mistral_raw = f"# Front page\n\n{mistral_front}\n\n# Back page\n\n{mistral_back}"

display(Markdown(mistral_raw))
print(f"Returned {len(mistral_raw):,} characters")
print(f"Completed in {mistral_seconds:.2f} seconds")

## 5. Azure Document Intelligence: layout Markdown

The `prebuilt-layout` model returns recognized content and document structure as Markdown. No GPT model restructures or corrects this text afterward.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat
from azure.core.credentials import AzureKeyCredential

document_intelligence_client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"].rstrip("/"),
    credential=AzureKeyCredential(os.environ["DOCUMENT_INTELLIGENCE_KEY"]),
)


def analyze_layout(path: Path) -> str:
    with path.open("rb") as document:
        poller = document_intelligence_client.begin_analyze_document(
            "prebuilt-layout",
            body=document,
            content_type="image/jpeg",
            output_content_format=DocumentContentFormat.MARKDOWN,
        )
    return poller.result().content


started = time.perf_counter()
document_intelligence_front = analyze_layout(FRONT_IMAGE)
document_intelligence_back = analyze_layout(BACK_IMAGE)
document_intelligence_seconds = time.perf_counter() - started
document_intelligence_raw = (
    f"# Front page\n\n{document_intelligence_front}"
    f"\n\n# Back page\n\n{document_intelligence_back}"
)

display(Markdown(document_intelligence_raw))
print(f"Returned {len(document_intelligence_raw):,} characters")
print(f"Completed in {document_intelligence_seconds:.2f} seconds")

## 6. Compare the outputs

Discuss these questions with your group:

1. Which approach preserved identifiers such as policy number, VIN, license plate, phone number, and police report number most accurately?
2. Which output makes transcription errors easiest to notice?
3. Which approach preserves the original document layout most clearly?
4. Which result is immediately usable by an application, and which results still require semantic field extraction?
5. Does the structured GPT output contain information that is missing from the OCR output, or did it infer anything unsupported?
6. Which approach would you choose for raw archival OCR, structured claim ingestion, and human review?

**Success criterion:** all three native outputs are visible, and your group can explain the strengths, weaknesses, and required downstream processing of each approach.